# Libraries

In [1]:
# Standard libraries
import json
import os

# Third-party libraries
import numpy as np
from tqdm import tqdm
import math, glob
import torch

# The goal is to take an original shard, process and save it

In [2]:
class _PreProcessor:
    """Batch-compatible preprocessor for storm nowcasting features (with `day` removed)."""
    def __init__(self, norm_json: str):
        with open(norm_json, "r") as f:
            self.norm = json.load(f)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        """
        This works for a batch/shard or a single file
        x: Tensor of shape (B, 140, 11) or (140, 11)
        returns Tensor of shape (B, 140, 10) or (140, 10)
        """

        # original header: [year, month, day, hour, minute, lat, lon, wp, tir, size, mask]

        # considered header: [month, hour, minute, lat, lon, wp, tir, size, mask]

        is_batch = x.dim() == 3  # (B, 140, 11)

        if not is_batch:
            x = x.unsqueeze(0)  # convert to batch shape

        # Drop 'year' (index 0) and 'day' (index 2 after dropping year)
        x = x[:, :, [1, 3, 4, 5, 6, 7, 8, 9, 10]]  # shape: (B, 140, 9)

        # Restarting index
        month  = x[:, :, 0]
        hour   = x[:, :, 1]
        minute = x[:, :, 2]
        lat    = x[:, :, 3]
        lon    = x[:, :, 4]
        wp     = x[:, :, 5]
        tir    = x[:, :, 6]
        size   = x[:, :, 7]
        mask   = x[:, :, 8]

        B, N = x.shape[:2]
        out = torch.empty((B, N, 10), dtype=torch.float32)

        # month encoding
        out[:, :, 0] = torch.sin(2 * math.pi * (month - 1) / 12.0)
        out[:, :, 1] = torch.cos(2 * math.pi * (month - 1) / 12.0)

        # time of day encoding
        tod = hour + minute / 60.0
        out[:, :, 2] = torch.sin(2 * math.pi * tod / 24.0)
        out[:, :, 3] = torch.cos(2 * math.pi * tod / 24.0)

        # lat, lon, wp, tir and size scaling
        out[:, :, 4] = (lat - self.norm["lat_min"]) / (self.norm["lat_max"] - self.norm["lat_min"])
        out[:, :, 5] = (lon - self.norm["lon_min"]) / (self.norm["lon_max"] - self.norm["lon_min"])
        out[:, :, 6] = torch.log1p(wp) / self.norm["wp_max"]
        out[:, :, 7] = (tir - self.norm["tir_min"]) / (self.norm["tir_max"] - self.norm["tir_min"])
        out[:, :, 8] = torch.log1p(size) / self.norm["size_max"]

        # mask kept as it is
        out[:, :, 9] = mask

        return out if is_batch else out.squeeze(0)

In [3]:
@torch.no_grad()
def process_single_shard(
    shards_dir: str,
    norm_path: str,
    output_dir: str,
    glob_pattern: str = "*.pt",
):
    """
    Preprocess a single shard file (selected using SLURM_ARRAY_TASK_ID)
    and save the processed data to `output_dir`.
    """
    # Load normalisation
    processor = _PreProcessor(norm_path)

    # Get SLURM task ID
    try:
        idx = int(os.environ["SLURM_ARRAY_TASK_ID"])
    except KeyError:
        raise RuntimeError("SLURM_ARRAY_TASK_ID not set. Use this function in an array job.")

    shard_files = sorted(glob.glob(os.path.join(shards_dir, glob_pattern)))
    if not shard_files:
        raise FileNotFoundError("No shard files found.")

    if idx >= len(shard_files):
        raise IndexError(f"SLURM_ARRAY_TASK_ID={idx} is out of range (found {len(shard_files)} files).")

    # Pick and process one file
    fp = shard_files[idx]
    data = torch.load(fp, map_location="cpu")
    x_batch, y_batch = data["inputs"], data["targets"]

    x_batch_proc = processor(x_batch.float())

    fname = os.path.basename(fp).replace(".pt", "_proc.pt")
    out_path = os.path.join(output_dir, fname)
    torch.save({"inputs": x_batch_proc, "targets": y_batch}, out_path)

    print(f"[{idx}] Processed: {fp} → {out_path}")

In [15]:
@torch.no_grad()
def process_file(
    file_path: str,
    norm_path: str,
    output_dir: str,
):
    """
    Preprocess a specific shard file by path and save the result.
    """
    processor = _PreProcessor(norm_path)

    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"Shard file not found: {file_path}")

    data = torch.load(file_path, map_location="cpu")
    if "inputs" not in data or "targets" not in data:
        raise KeyError(f"Missing 'inputs' or 'targets' in: {file_path}")

    x_batch, y_batch = data["inputs"], data["targets"]
    x_batch_proc = processor(x_batch.float())

    os.makedirs(output_dir, exist_ok=True)
    fname = os.path.basename(file_path).replace(".pt", "_proc.pt")
    out_path = os.path.join(output_dir, fname)
    torch.save({"inputs": x_batch_proc, "targets": y_batch}, out_path)

    print(f"Processed: {file_path} → {out_path}")


In [5]:
LEAD_TIME = "1"
PARTITION = "train"

In [17]:
root_dir = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/t{LEAD_TIME}/{PARTITION}"
norm_path = "/home/users/mendrika/EPS-Impact-Case-AI-Nowcasting/model/africa/normalisation/parameters/normalisation.json"
output_path = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/processed/t{LEAD_TIME}/{PARTITION}"

In [18]:
process_file(
    file_path=root_dir[0],
    norm_path=norm_path,
    output_dir=output_path
)

FileNotFoundError: Shard file not found: /

In [ ]:
process_single_shard(
    shards_dir=root_dir,
    norm_path=norm_path,
    output_dir=output_path
)

TypeError: process_file() got an unexpected keyword argument 'shards_dir'